# 04 — Activation extraction → emotion vectors + cumulative timeseries

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

This notebook turns the stories from **nb03** into the two artefacts the rest of the pipeline needs. It runs each story through a **target model**, reads the residual stream, and produces:

1. **Emotion vectors** — one direction per emotion, by **averaging** the pooled per-story activations and taking a **difference-of-means** against a neutral baseline, then **k-means clustering** the directions. → `emotion_vectors.npz`
2. **Per-step ("cumulative-feeding") story timeseries** — for a sample of stories, the model's representation **at each step of reading the story**: the features after consuming the prefix `tokens[:t+1]` for every `t` (`"I"`, `"I think"`, `"I think we"`, …). Stored as a `(seq, n_layers, d_model)` array per story — **token × layer × feature** — preserving the full temporal progression. nb05 projects the emotion vector onto this to show emotion being "loaded" token-by-token. → `cumulative_timeseries.npz`

   > **Why one forward pass suffices.** These are *causal* decoder LMs: the hidden state at position `t` in a single full-sequence pass is identical to the last-token hidden state you'd get by feeding only `tokens[:t+1]` (causal masking → position `t` never sees the future; positions don't shift). So the per-step timeseries is exactly the per-token residual `resid[:, t, :]` — extracted in *one* pass, not the running average of it. A literal prefix-refeed mode (`EMOVEC_CUM_METHOD=refeed`) is available for non-causal models or paranoia; it's `O(seq²)` and, for causal models, returns the same numbers up to BPE-boundary effects.

Along the way it also saves the raw **pooled segment features** (`segment_features.npz`) so vectors can be recomputed without re-running the model.

> **Target model ≠ generator model.** nb03's *generator* wrote the stories; nb04's *target* is the model whose internal emotion representation we are extracting. They can differ (e.g. generate with Qwen-7B, probe GPT-2). Default target is **`gpt2`** so this runs in seconds on any GPU/CPU for development — swap via `EMOVEC_TARGET_MODEL`.

**Demo / development mode (default).** The notebook uses *whatever stories are already on disk* — it discovers the largest `stories.jsonl`, reports how many stories exist and their emotion-label distribution, and caps work so a partial generation run still yields a full set of (rough) vectors and figures. Set `EMOVEC_DEMO=0` for a full run.

**Runs head-less.** Every knob is an environment variable, so the same notebook drives a Colab T4 demo or an HPC `jupyter nbconvert --execute` / `papermill` run on a big target model.

## 0 — Setup

Same portable bootstrap as nb02/nb03 (Drive on Colab, `EMOVEC_WORK_DIR` on HPC/local) plus the extraction stack (`torch`, `transformers`; `transformer_lens` only if you pick the `tlens` backend; `scikit-learn` for clustering/PCA).

| Env var | Effect | Default |
|---|---|---|
| `EMOVEC_WORK_DIR` | Where stories live / features are written. | Drive on Colab, repo root locally |
| `EMOVEC_MOUNT_DRIVE` | Mount Google Drive on Colab. | `1` on Colab |
| `HF_TOKEN` | HF token for gated target models. | Colab Secrets → env |
| `EMOVEC_TARGET_MODEL` | Model to probe (≠ generator). | `gpt2` |
| `EMOVEC_BACKEND` | `hf` (universal) / `tlens` / `auto`. | `hf` |
| `EMOVEC_PRECISION` | `auto`/`bf16`/`fp16`/`4bit`/`8bit` (HF backend). | `auto` |
| `EMOVEC_DEVICE_MAP` | HF device map (`auto` spreads GPUs). | `auto` |

In [ ]:
# Dependencies. Installed on Colab; assumed present on a provisioned HPC env.
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30",
                    "bitsandbytes>=0.43", "scikit-learn", "tqdm"], check=False)


In [ ]:
import json, os, re, time, hashlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

def _env(name, default):
    v = os.environ.get(name)
    return v if v not in (None, "") else default

def _env_int(name, default):
    v = os.environ.get(name)
    return int(v) if v not in (None, "") else default

def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")

# ── Hugging Face token (gated target models). env var > Colab Secrets. ───────
if "HF_TOKEN" not in os.environ and IN_COLAB:
    try:
        from google.colab import userdata
        _t = userdata.get("HF_TOKEN")
        if _t:
            os.environ["HF_TOKEN"] = _t
    except Exception:
        pass

# ── Working directory (must match nb02/nb03 so we find the stories) ──────────
MOUNT_DRIVE = _env_flag("EMOVEC_MOUNT_DRIVE", IN_COLAB)
if os.environ.get("EMOVEC_WORK_DIR"):
    WORK_DIR = Path(os.environ["EMOVEC_WORK_DIR"]).expanduser()
elif IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/EmoVecLLM")
elif IN_COLAB:
    WORK_DIR = Path("/content/EmoVecLLM")
else:
    WORK_DIR = Path("..").resolve()

DATA_DIR = WORK_DIR / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    N_GPU = torch.cuda.device_count()
else:
    DEVICE, GPU_NAME, VRAM_GB, N_GPU = "cpu", "cpu", 0.0, 0

print(f"IN_COLAB={IN_COLAB}  WORK_DIR={WORK_DIR}")
print(f"device  : {DEVICE}  ({GPU_NAME}, {VRAM_GB:.0f} GB x {N_GPU} GPU)")


## 1 — Config

The two lines you usually touch are `TARGET_MODEL` and `DEMO`. Everything else auto-adapts but is overridable for a cluster run.

| Env var | Effect | Default |
|---|---|---|
| `EMOVEC_DEMO` | Dev mode: cap stories, fast target, rough vectors. | `1` |
| `EMOVEC_TARGET_MODEL` | Model to probe. | `gpt2` |
| `EMOVEC_POOL` | Per-story token pooling: `mean` / `last`. | `mean` |
| `EMOVEC_CLUSTER_LAYER` | Layer used for clustering/geometry (`mid` or int). | `mid` |
| `EMOVEC_CUM_LAYER` | Layer used for the loading projection in nb05 (`mid`/int). | `mid` |
| `EMOVEC_CUM_LAYERS` | Layers stored per-step: int `N` (evenly spaced) / `all` / `mid` / csv. | `8` |
| `EMOVEC_CUM_METHOD` | `singlepass` (causal-exact) or `refeed` (literal prefixes). | `singlepass` |
| `EMOVEC_K` | k-means clusters over emotion vectors. | `10` |
| `EMOVEC_BASELINE` | `auto`/`neutral_mean`/`global_mean`/`project_pcs`/`none`. | `auto` |
| `EMOVEC_BASELINE_PCS` | PCs removed when `project_pcs`. | `5` |
| `EMOVEC_MAX_SEG_PER_EMOTION` | Cap stories/emotion (`0`=all). | demo `8`, else `0` |
| `EMOVEC_MAX_TIMESERIES` | How many stories get a saved cumulative timeseries. | demo `60`, else `400` |
| `EMOVEC_MIN_TOKENS` / `EMOVEC_MAX_TOKENS` | Per-story token bounds. | `16` / `256` |
| `EMOVEC_STORIES_PATH` | Explicit `stories.jsonl` (skip auto-discovery). | auto |

In [ ]:
DEMO = _env_flag("EMOVEC_DEMO", True)

TARGET_MODEL = _env("EMOVEC_TARGET_MODEL", "gpt2")
BACKEND      = _env("EMOVEC_BACKEND", "hf")        # hf | tlens | auto
PRECISION    = _env("EMOVEC_PRECISION", "auto")
DEVICE_MAP   = _env("EMOVEC_DEVICE_MAP", "auto")

POOL          = _env("EMOVEC_POOL", "mean")        # mean | last
CLUSTER_LAYER = _env("EMOVEC_CLUSTER_LAYER", "mid")
CUM_LAYER     = _env("EMOVEC_CUM_LAYER", "mid")    # projection layer for nb05's loading view
CUM_LAYERS    = _env("EMOVEC_CUM_LAYERS", "8")     # per-step storage: N even | all | mid | csv
CUM_METHOD    = _env("EMOVEC_CUM_METHOD", "singlepass")  # singlepass (causal-exact) | refeed
K             = _env_int("EMOVEC_K", 10)
BASELINE      = _env("EMOVEC_BASELINE", "auto")    # see §6
BASELINE_PCS  = _env_int("EMOVEC_BASELINE_PCS", 5)

MIN_TOKENS = _env_int("EMOVEC_MIN_TOKENS", 16)
MAX_TOKENS = _env_int("EMOVEC_MAX_TOKENS", 256)

# Demo defaults keep a partial run cheap; override any of them explicitly.
MAX_SEG_PER_EMOTION = _env_int("EMOVEC_MAX_SEG_PER_EMOTION", 8 if DEMO else 0)
MAX_TIMESERIES      = _env_int("EMOVEC_MAX_TIMESERIES", 60 if DEMO else 400)

print(f"DEMO={DEMO}   target={TARGET_MODEL}   backend={BACKEND}")
print(f"pool={POOL}   cluster_layer={CLUSTER_LAYER}   k={K}")
print(f"per-step timeseries: layers={CUM_LAYERS}  method={CUM_METHOD}  proj_layer={CUM_LAYER}")
print(f"baseline={BASELINE} (pcs={BASELINE_PCS})")
print(f"max_seg/emotion={MAX_SEG_PER_EMOTION or 'all'}   max_timeseries={MAX_TIMESERIES}")
print(f"tokens in [{MIN_TOKENS}, {MAX_TOKENS}]")


## 2 — Find the stories and report what's available

Demo-mode discovery: if `EMOVEC_STORIES_PATH` isn't set, we scan `data/processed/stories/**/stories.jsonl` and take the file with the **most** records (i.e. whichever generation run is furthest along). We then flatten each generation call into its individual **segments** (one story per row), drop too-short/too-long ones, and print the **emotion-label distribution** — the key sanity check on a partial run.

In [ ]:
STORIES_PATH = _env("EMOVEC_STORIES_PATH", "")
if STORIES_PATH:
    stories_file = Path(STORIES_PATH).expanduser()
else:
    candidates = sorted((DATA_DIR / "stories").rglob("stories.jsonl"))
    assert candidates, (f"no stories.jsonl under {DATA_DIR/'stories'} — run nb03 first, "
                        f"or set EMOVEC_STORIES_PATH")
    def _nlines(p):
        try:
            return sum(1 for _ in p.open())
        except Exception:
            return 0
    stories_file = max(candidates, key=_nlines)
    if len(candidates) > 1:
        print(f"found {len(candidates)} stories.jsonl; using the largest:")
print(f"stories file : {stories_file}")

records = [json.loads(l) for l in stories_file.read_text().splitlines() if l.strip()]
SPEC_HASH      = records[0].get("spec_hash", "nospec") if records else "nospec"
GENERATOR_USED = records[0].get("generator_model", "?") if records else "?"
print(f"spec_hash    : {SPEC_HASH}   (stories generated by {GENERATOR_USED})")
print(f"job outputs  : {len(records)}")


In [ ]:
# Flatten generation calls -> individual segments (one story per item).
def _approx_ntok(s):
    return len(s.split())

items = []          # each: dict(text, emotion, kind, job_id, topic, seg_idx)
for r in records:
    segs = r.get("segments") or ([r["raw"]] if r.get("raw") else [])
    for si, seg in enumerate(segs):
        seg = (seg or "").strip()
        nt = _approx_ntok(seg)
        if nt < MIN_TOKENS:
            continue
        items.append({
            "text": seg,
            "emotion": r.get("emotion"),
            "kind": r.get("kind"),
            "job_id": r.get("job_id"),
            "topic": r.get("topic"),
            "seg_idx": si,
        })

# Per-emotion cap (emotion stories only); baseline kinds are kept uncapped.
if MAX_SEG_PER_EMOTION:
    per = defaultdict(int)
    capped = []
    for it in items:
        if it["kind"] == "emotion_story":
            if per[it["emotion"]] >= MAX_SEG_PER_EMOTION:
                continue
            per[it["emotion"]] += 1
        capped.append(it)
    items = capped

emo_items     = [it for it in items if it["kind"] == "emotion_story"]
neutral_items = [it for it in items if it["kind"] in ("neutral_dialogue", "neutral_story")]
emo_dist = Counter(it["emotion"] for it in emo_items)

print(f"usable segments     : {len(items)}  "
      f"(emotion={len(emo_items)}, neutral={len(neutral_items)}, "
      f"other={len(items)-len(emo_items)-len(neutral_items)})")
print(f"emotions covered    : {len(emo_dist)}")
print(f"segments/emotion    : min={min(emo_dist.values()) if emo_dist else 0}  "
      f"max={max(emo_dist.values()) if emo_dist else 0}  "
      f"mean={np.mean(list(emo_dist.values())):.1f}" if emo_dist else "n/a")
print(f"neutral baseline    : {len(neutral_items)} segments "
      f"({'available' if neutral_items else 'MISSING -> will fall back to global mean'})")
print()
print("kind breakdown:")
for k, c in sorted(Counter(it['kind'] for it in items).items()):
    print(f"  {k:<20} {c:>5}")
print()
print("top / bottom emotions by coverage:")
ranked = emo_dist.most_common()
for e, c in ranked[:5]:
    print(f"  {e:<22} {c}")
if len(ranked) > 10:
    print("  ...")
    for e, c in ranked[-5:]:
        print(f"  {e:<22} {c}")


## 3 — Load the target model (swappable)

`extract_resid(text)` returns `(resid, tokens)` with `resid` shaped **(n_layers, seq, d_model)** — the post-block residual stream at every layer — regardless of backend:

- **`hf`** (default, universal): `AutoModelForCausalLM(output_hidden_states=True)`. Works for *any* HF model, and reuses nb03's precision/quantization loader so a 7-8B target fits a T4 in 4-bit or shards across GPUs on HPC via `device_map="auto"`. Hidden state 0 (embeddings) is dropped so layer indices line up with blocks.
- **`tlens`**: TransformerLens `HookedTransformer` + `run_with_cache` (`blocks.{i}.hook_resid_post`), matching nb01. Cleaner hooks for steering work later, but only covers models TransformerLens supports.

In [ ]:
def _resolve_precision(pref, vram_gb):
    if pref != "auto":
        return pref
    if DEVICE == "cpu":
        return "fp32"
    return "4bit" if (vram_gb and vram_gb < 24) else "bf16"

backend = BACKEND
tl_model = hf_model = hf_tok = None

if backend in ("tlens", "auto"):
    try:
        from transformer_lens import HookedTransformer
        tl_model = HookedTransformer.from_pretrained(TARGET_MODEL, device=DEVICE)
        tl_model.eval()
        backend = "tlens"
    except Exception as e:
        if BACKEND == "tlens":
            raise
        print(f"tlens unavailable for {TARGET_MODEL} ({type(e).__name__}); using hf backend.")
        backend = "hf"

if backend == "hf" or tl_model is None:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    try:
        from transformers import BitsAndBytesConfig
    except Exception:
        BitsAndBytesConfig = None
    prec = _resolve_precision(PRECISION, VRAM_GB)
    hf_token = os.environ.get("HF_TOKEN")
    hf_tok = AutoTokenizer.from_pretrained(TARGET_MODEL, trust_remote_code=True, token=hf_token)
    kw = dict(trust_remote_code=True, token=hf_token, output_hidden_states=True)
    if DEVICE == "cpu":
        kw.update(torch_dtype=torch.float32)
    elif prec in ("4bit", "8bit") and BitsAndBytesConfig is not None:
        kw.update(quantization_config=BitsAndBytesConfig(
            load_in_4bit=(prec == "4bit"), load_in_8bit=(prec == "8bit"),
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True), device_map=DEVICE_MAP)
    else:
        dtype = (torch.bfloat16 if prec == "bf16" and torch.cuda.is_bf16_supported()
                 else torch.float16)
        kw.update(torch_dtype=dtype, device_map=DEVICE_MAP)
    hf_model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, **kw)
    hf_model.eval()
    backend = "hf"

print(f"backend = {backend}   target = {TARGET_MODEL}")


In [ ]:
@torch.no_grad()
def extract_resid(text):
    'Return (resid (n_layers, seq, d_model) float32, tokens list[str]).'
    if backend == "tlens":
        toks = tl_model.to_tokens(text, prepend_bos=True)[:, :MAX_TOKENS]
        _, cache = tl_model.run_with_cache(toks, remove_batch_dim=True)
        n_layers = tl_model.cfg.n_layers
        resid = np.stack([cache[f"blocks.{i}.hook_resid_post"].cpu().float().numpy()
                          for i in range(n_layers)], axis=0)
        tok_strs = tl_model.to_str_tokens(toks)
        return resid.astype(np.float32), list(tok_strs)
    # hf backend
    enc = hf_tok(text, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)
    dev = hf_model.get_input_embeddings().weight.device
    enc = {k: v.to(dev) for k, v in enc.items()}
    out = hf_model(**enc)
    hs = out.hidden_states                  # tuple len n_layers+1, each (1, seq, d)
    resid = np.stack([h[0].cpu().float().numpy() for h in hs[1:]], axis=0)  # drop embeddings
    tok_strs = hf_tok.convert_ids_to_tokens(enc["input_ids"][0].tolist())
    return resid.astype(np.float32), tok_strs

# Probe shapes on one segment.
_probe_text = (emo_items or items)[0]["text"]
_r, _t = extract_resid(_probe_text)
N_LAYERS, _, D_MODEL = _r.shape
def _resolve_layer(spec):
    return N_LAYERS // 2 if str(spec) == "mid" else int(spec)
CLUSTER_L = _resolve_layer(CLUSTER_LAYER)
CUM_L     = _resolve_layer(CUM_LAYER)

def _resolve_layer_set(spec, n_layers, must_include):
    spec = str(spec)
    if spec == "all":
        idx = list(range(n_layers))
    elif spec == "mid":
        idx = [n_layers // 2]
    elif spec.lstrip("-").isdigit():                      # N evenly-spaced layers
        k = min(int(spec), n_layers)
        idx = sorted(set(np.linspace(0, n_layers - 1, k).round().astype(int).tolist()))
    else:                                                  # explicit csv of indices
        idx = [int(x) for x in spec.split(",")]
    return sorted(set(idx) | {must_include})              # always keep the projection layer

CUM_LAYER_IDX = _resolve_layer_set(CUM_LAYERS, N_LAYERS, CUM_L)
print(f"resid per story : (n_layers={N_LAYERS}, seq<= {MAX_TOKENS}, d_model={D_MODEL})")
print(f"cluster layer   : {CLUSTER_L}   projection layer: {CUM_L}")
print(f"per-step layers : {CUM_LAYER_IDX}  ({len(CUM_LAYER_IDX)} of {N_LAYERS})")


## 4 — Numeric core (pure NumPy)

These functions carry the actual method and are deliberately backend-free so they can be reasoned about (and unit-tested) without a model:

- `pool_segment` collapses a story's `(n_layers, seq, d)` residual to one `(n_layers, d)` vector — `mean` over tokens (Anthropic-style, full passage) or `last` token. (Used for the emotion *vectors*.)
- The **per-step timeseries** is not a function here — it's just the per-token residual `resid[layers].transpose → (seq, n_sel_layers, d)`, read out in §5. Row *t* is the model's state after reading `tokens[:t+1]` (the cumulation is done by the model's attention, so no averaging is applied).
- `build_emotion_vectors` averages pooled vectors per emotion and applies the chosen **baseline** (see §6): plain difference-of-means, project-out-PCs, or none.
- `project_out_pcs` removes the top-N principal directions of the neutral baseline from every vector (Anthropic's trick to strip the dominant style direction).
- `cosine_sim_matrix` is the geometry primitive nb05 visualises.

In [ ]:
def pool_segment(resid, pool="mean"):
    'resid (n_layers, seq, d) -> (n_layers, d).'
    if pool == "last":
        return resid[:, -1, :].astype(np.float32)
    return resid.mean(axis=1).astype(np.float32)

def _unit(v, axis=-1, eps=1e-12):
    return v / (np.linalg.norm(v, axis=axis, keepdims=True) + eps)

def project_out_pcs(X, basis):
    'Remove the span of `basis` (k, d) rows from each row of X (n, d).'
    if basis is None or len(basis) == 0:
        return X
    B = _unit(np.asarray(basis, dtype=np.float64), axis=1)   # orthonormal-ish rows
    # Gram-Schmidt to be safe (few components).
    Q = []
    for b in B:
        for q in Q:
            b = b - (b @ q) * q
        n = np.linalg.norm(b)
        if n > 1e-9:
            Q.append(b / n)
    if not Q:
        return X
    Q = np.stack(Q, axis=0)                                  # (k, d)
    return (X - (X @ Q.T) @ Q).astype(np.float32)

def cosine_sim_matrix(X):
    'X (n, d) -> (n, n) cosine similarity.'
    Xn = _unit(np.asarray(X, dtype=np.float64), axis=1)
    return (Xn @ Xn.T).astype(np.float32)

def build_emotion_vectors(class_means, neutral_mean, global_mean, mode,
                          neutral_pcs=None):
    '''class_means: dict[emotion] -> (n_layers, d). Returns (emotions, vectors).
    vectors[e] is the per-layer emotion direction under `mode`.'''
    emotions = sorted(class_means)
    M = np.stack([class_means[e] for e in emotions], axis=0)   # (E, n_layers, d)
    if mode == "none":
        V = M
    elif mode == "global_mean":
        V = M - global_mean[None]
    else:   # neutral_mean / project_pcs both centre on neutral first
        base = neutral_mean if neutral_mean is not None else global_mean
        V = M - base[None]
        if mode == "project_pcs" and neutral_pcs is not None:
            V = np.stack([project_out_pcs(V[:, L, :], neutral_pcs[L])
                          for L in range(V.shape[1])], axis=1)
    return emotions, V.astype(np.float32)


## 5 — Extract activations

One forward pass per segment gives the full `(n_layers, seq, d)` residual. From it we take:

- the **pooled** `(n_layers, d)` vector for *every* segment (cheap → the emotion vectors), and
- for up to `MAX_TIMESERIES` segments, the **per-step timeseries** `(seq, n_sel_layers, d)` — the residual at each token, i.e. the model's state after reading every prefix. Stored as `float16` and at `CUM_LAYER_IDX` layers only, so the artefact stays bounded even for big targets.

`singlepass` (default) slices the per-token residual from the one pass we already ran (causal-exact, free). `refeed` literally re-runs the model on each growing prefix and reads the last token — `O(seq²)`, for non-causal models or as a sanity check. Emotion segments are prioritised for the timeseries, spread across emotions.

In [ ]:
from tqdm.auto import tqdm

# Choose which segments get a saved cumulative timeseries (spread over emotions).
ts_pick = set()
if MAX_TIMESERIES and emo_items:
    by_emo = defaultdict(list)
    for i, it in enumerate(items):
        if it["kind"] == "emotion_story":
            by_emo[it["emotion"]].append(i)
    order, ptr = list(by_emo), 0
    while len(ts_pick) < MAX_TIMESERIES and order:
        e = order[ptr % len(order)]
        if by_emo[e]:
            ts_pick.add(by_emo[e].pop(0))
        else:
            order.remove(e)
            continue
        ptr += 1

@torch.no_grad()
def refeed_features(text, layer_idx):
    'Literal prefix re-feed: row t = last-token residual after reading tokens[:t+1].'
    if backend == "tlens":
        ids = tl_model.to_tokens(text, prepend_bos=True)[0, :MAX_TOKENS]
        toks = tl_model.to_str_tokens(ids)
    else:
        ids = hf_tok(text, truncation=True, max_length=MAX_TOKENS,
                     return_tensors="pt")["input_ids"][0]
        toks = hf_tok.convert_ids_to_tokens(ids.tolist())
    rows = []
    for t in range(1, len(ids) + 1):
        sub = ids[:t]
        if backend == "tlens":
            _, c = tl_model.run_with_cache(sub.unsqueeze(0), remove_batch_dim=True)
            rows.append(np.stack([c[f"blocks.{L}.hook_resid_post"][-1].cpu().float().numpy()
                                  for L in layer_idx]))
        else:
            dev = hf_model.get_input_embeddings().weight.device
            hs = hf_model(input_ids=sub.unsqueeze(0).to(dev)).hidden_states
            rows.append(np.stack([hs[L + 1][0, -1].cpu().float().numpy() for L in layer_idx]))
    return np.stack(rows, axis=0), toks            # (seq, n_sel, d), tokens

def per_step_features(resid, text, layer_idx):
    'Return (seq, n_sel_layers, d): the residual at each token across selected layers.'
    if CUM_METHOD == "refeed":
        feats, toks = refeed_features(text, layer_idx)
        return feats, toks
    # singlepass (causal-exact): slice the per-token residual we already have.
    return resid[layer_idx].transpose(1, 0, 2), None    # (seq, n_sel, d)

X = np.zeros((len(items), N_LAYERS, D_MODEL), dtype=np.float32)
n_tokens = np.zeros(len(items), dtype=np.int32)
cum_list, cum_meta = [], []          # ragged per-step timeseries + metadata

t0 = time.time()
for i, it in enumerate(tqdm(items, desc="extract")):
    resid, toks = extract_resid(it["text"])
    n_tokens[i] = resid.shape[1]
    X[i] = pool_segment(resid, POOL)
    if i in ts_pick:
        feats, toks2 = per_step_features(resid, it["text"], CUM_LAYER_IDX)
        cum_list.append(feats.astype(np.float16))           # (seq, n_sel_layers, d)
        cum_meta.append({"emotion": it["emotion"], "kind": it["kind"],
                         "job_id": it["job_id"], "topic": it["topic"],
                         "tokens": (toks2 or toks)[:feats.shape[0]]})
dt = time.time() - t0
print(f"extracted {len(items)} segments in {dt/60:.1f} min "
      f"({len(items)/max(dt,1):.1f} seg/s); {len(cum_list)} per-step timeseries kept "
      f"(shape (seq, {len(CUM_LAYER_IDX)}, {D_MODEL}))")


## 6 — Neutral baseline → emotion vectors → clusters

### How we normalise the embedding space (the neutral question)

Raw class means are dominated by what *all* generations share (topic words, "story-ness", model boilerplate). To isolate the **emotion** direction we subtract a neutral baseline. Four modes (`EMOVEC_BASELINE`):

| Mode | What it does | When |
|---|---|---|
| `neutral_mean` | `vec_e = mean_e − mean(neutral)` — classic difference-of-means. | neutral segments present |
| `project_pcs` | as above, then project out the **top-N PCs of the neutral set** per layer — removes the dominant *style* axis Anthropic warns about. | strong style confound |
| `global_mean` | subtract the mean over **all** segments. | **no** neutral set (demo fallback) |
| `none` | raw class means. | diagnostics only |

`auto` picks `neutral_mean` if a neutral set exists, else `global_mean` (and warns).

### ⚠️ Neutral *dialogue* vs neutral *story* — a baseline decision still open

Anthropic's baseline is a neutral **Human/Assistant dialogue**. Our emotion items are **story prose**. That mismatch means the difference-of-means picks up a *story-vs-dialogue style* direction on top of emotion — hence `project_pcs` to strip it.

**We may instead generate neutral *stories*** (same form, emotion held neutral). Implications, baked into this notebook so the switch is one config change:

- A **style-matched** baseline means `neutral_mean` alone should largely suffice — `project_pcs` becomes optional/diagnostic rather than necessary.
- But a *too-similar* baseline risks subtracting genuine shared narrative content (plot, characters), shrinking real emotion signal. Watch the vector norms and held-out separability (nb06) when you switch.
- nb04 already treats any `kind` in `{neutral_dialogue, neutral_story}` as baseline, so **dropping neutral-story rows into nb03's manifest is all that's needed** — no code change here. The active baseline form is recorded in `emotion_vectors.npz["baseline_kinds"]` for provenance.

In [ ]:
# Class means per emotion (emotion stories only).
class_segs = defaultdict(list)
for i, it in enumerate(items):
    if it["kind"] == "emotion_story" and it["emotion"]:
        class_segs[it["emotion"]].append(i)
class_means = {e: X[idx].mean(axis=0) for e, idx in class_segs.items()}
n_per_emotion = {e: len(idx) for e, idx in class_segs.items()}

global_mean = X.mean(axis=0)                                   # (n_layers, d)
neutral_idx = [i for i, it in enumerate(items)
               if it["kind"] in ("neutral_dialogue", "neutral_story")]
neutral_mean = X[neutral_idx].mean(axis=0) if neutral_idx else None
baseline_kinds = sorted({items[i]["kind"] for i in neutral_idx})

# Resolve auto baseline.
mode = BASELINE
if mode == "auto":
    mode = "neutral_mean" if neutral_idx else "global_mean"
if mode in ("neutral_mean", "project_pcs") and not neutral_idx:
    print("no neutral segments -> falling back to global_mean baseline")
    mode = "global_mean"

# Top-N neutral PCs per layer (only if needed).
neutral_pcs = None
if mode == "project_pcs":
    from sklearn.decomposition import PCA
    Xn = X[neutral_idx]
    neutral_pcs = []
    for L in range(N_LAYERS):
        k = min(BASELINE_PCS, max(1, Xn.shape[0] - 1))
        neutral_pcs.append(PCA(n_components=k).fit(Xn[:, L, :] - Xn[:, L, :].mean(0)).components_)

emotions, V = build_emotion_vectors(class_means, neutral_mean, global_mean,
                                    mode, neutral_pcs)
print(f"baseline mode   : {mode}   (baseline kinds: {baseline_kinds or 'none'})")
print(f"emotion vectors : {V.shape}  (emotions x n_layers x d_model)")
print(f"emotions        : {len(emotions)}")


In [ ]:
# Cluster the emotion vectors at the chosen layer.
from sklearn.cluster import KMeans

Vc = V[:, CLUSTER_L, :]                       # (E, d)
k_eff = int(min(K, max(2, len(emotions))))
if len(emotions) >= 2:
    km = KMeans(n_clusters=k_eff, n_init=10, random_state=0).fit(_unit(Vc))
    cluster_labels = km.labels_
else:
    cluster_labels = np.zeros(len(emotions), dtype=int)
print(f"k-means: {k_eff} clusters over {len(emotions)} emotion vectors at layer {CLUSTER_L}")
for c in range(k_eff):
    members = [emotions[i] for i in range(len(emotions)) if cluster_labels[i] == c]
    if members:
        print(f"  cluster {c}: {', '.join(members[:8])}{' ...' if len(members) > 8 else ''}")


## 7 — Save features

Three artefacts under `data/processed/features/{spec_hash}/{target_model}/`, all keyed by spec + target model so different targets never clash:

- `segment_features.npz` — pooled `(n_seg, n_layers, d)` + labels (recompute vectors without the model).
- `emotion_vectors.npz` — the directions, baseline, clusters, provenance.
- `cumulative_timeseries.npz` — ragged per-step timeseries: one `(seq, n_sel_layers, d)` float16 array per sampled story (+ tokens, `cum_layer_indices`, `cum_proj_layer`), for nb05's loading curves.

In [ ]:
def _safe(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)

OUT_DIR = DATA_DIR / "features" / SPEC_HASH / _safe(TARGET_MODEL)
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT_DIR / "segment_features.npz",
    X=X, n_tokens=n_tokens,
    emotions=np.array([it["emotion"] for it in items], dtype=object),
    kinds=np.array([it["kind"] for it in items], dtype=object),
    job_ids=np.array([it["job_id"] for it in items], dtype=object),
    topics=np.array([it["topic"] for it in items], dtype=object),
    pool=POOL, target_model=TARGET_MODEL, spec_hash=SPEC_HASH,
    n_layers=N_LAYERS, d_model=D_MODEL,
)

np.savez_compressed(
    OUT_DIR / "emotion_vectors.npz",
    vectors=V,
    emotions=np.array(emotions, dtype=object),
    class_means=np.stack([class_means[e] for e in emotions], axis=0),
    n_per_emotion=np.array([n_per_emotion[e] for e in emotions]),
    baseline_mean=(neutral_mean if neutral_mean is not None else global_mean),
    global_mean=global_mean,
    baseline_mode=mode,
    baseline_kinds=np.array(baseline_kinds, dtype=object),
    cluster_labels=cluster_labels, cluster_layer=CLUSTER_L, k=k_eff,
    cum_layer=CUM_L, target_model=TARGET_MODEL, spec_hash=SPEC_HASH,
)

# Ragged per-step timeseries: each entry is (seq, n_sel_layers, d) float16.
# cum_layer_indices says which model layers the 2nd axis corresponds to;
# cum_proj_layer is the one nb05 projects emotion vectors onto.
np.savez_compressed(
    OUT_DIR / "cumulative_timeseries.npz",
    cum=np.array(cum_list, dtype=object),
    cum_emotions=np.array([m["emotion"] for m in cum_meta], dtype=object),
    cum_kinds=np.array([m["kind"] for m in cum_meta], dtype=object),
    cum_job_ids=np.array([m["job_id"] for m in cum_meta], dtype=object),
    cum_topics=np.array([m["topic"] for m in cum_meta], dtype=object),
    cum_tokens=np.array([m["tokens"] for m in cum_meta], dtype=object),
    cum_layer_indices=np.array(CUM_LAYER_IDX),
    cum_proj_layer=CUM_L, cum_method=CUM_METHOD,
    target_model=TARGET_MODEL, spec_hash=SPEC_HASH,
)

manifest = {
    "spec_hash": SPEC_HASH, "target_model": TARGET_MODEL, "backend": backend,
    "stories_file": str(stories_file), "generator_model": GENERATOR_USED,
    "n_segments": len(items), "n_emotions": len(emotions),
    "n_neutral": len(neutral_idx), "n_timeseries": len(cum_list),
    "pool": POOL, "baseline_mode": mode, "baseline_kinds": baseline_kinds,
    "n_layers": int(N_LAYERS), "d_model": int(D_MODEL),
    "cluster_layer": int(CLUSTER_L), "cum_proj_layer": int(CUM_L),
    "cum_layer_indices": [int(x) for x in CUM_LAYER_IDX],
    "cum_method": CUM_METHOD, "k": int(k_eff),
    "demo": bool(DEMO), "updated": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
(OUT_DIR / "features_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"saved -> {OUT_DIR}")
print(json.dumps(manifest, indent=2))


## 8 — Sanity check + next

A quick internal check that the vectors carry *emotion-specific* signal: each emotion's own vector should be its nearest neighbour among emotions more often than chance. On a tiny demo run this will be noisy — it sharpens as coverage grows. **nb05** loads these three files and draws the preliminary figures (distribution, vector geometry, and the emotion-loading curves).

In [ ]:
S = cosine_sim_matrix(V[:, CLUSTER_L, :])
np.fill_diagonal(S, -np.inf)
nn = S.argmax(axis=1)
# Nearest-neighbour cluster agreement (do similar emotions cluster together?).
agree = np.mean([cluster_labels[i] == cluster_labels[nn[i]] for i in range(len(emotions))])
print(f"nearest-neighbour shares cluster: {100*agree:.0f}%  "
      f"(chance ~ {100/max(k_eff,1):.0f}%)")
print()
print("a few nearest-neighbour emotion pairs (layer {}):".format(CLUSTER_L))
for i in range(min(8, len(emotions))):
    print(f"  {emotions[i]:<20} -> {emotions[nn[i]]}")
print()
print("Run nb05 for the figures. To go beyond demo: EMOVEC_DEMO=0, a real target "
      "(e.g. EMOVEC_TARGET_MODEL=EleutherAI/pythia-1.4b), and full story coverage.")
